In [ ]:
# Vector store for storing/searching embeddings
pip install llama-index-vector-stores-lancedb

# OpenAI multimodal LLM integration for LlamaIndex
pip install llama-index-multi-modal-llms-openai

# CLIP embeddings for image/text similarity
pip install llama-index-embeddings-clip

# File/document readers (PDF, DOCX, etc.)
pip install llama-index-readers-file

# Core LlamaIndex framework
pip install llama-index

# OpenAI Whisper for speech-to-text
pip install -U openai-whisper

# LanceDB vector database
pip install lancedb

# Video editing/processing
pip install moviepy

# Download YouTube videos
pip install pytube

# Audio processing/manipulation
pip install pydub

# Speech recognition / audio-to-text utilities
pip install SpeechRecognition

# FFmpeg integration for audio/video processing
pip install ffmpeg-python

# Read/write audio files
pip install soundfile

# PyTorch + computer vision/deep-learning utilities
pip install torch torchvision

# Plotting and visualization
pip install matplotlib

# Image processing and computer vision algorithms
pip install scikit-image

# Text normalization utilities used by CLIP
pip install ftfy

# Regular-expression support
pip install regex

# Progress bars for long-running operations
pip install tqdm

In [ ]:
from moviepy import videofileclip
from pathlib import path
import SpeechRecognition as sr
from pytube import YouTube
from pprint import pprint
from pil import image
import matplotlib.pyplot as plt

In [ ]:
from google import genai
client = genai.client(
    api_key="api_key"
)

In [ ]:
video_url=" "
output_video_path="./data/video_data "
#I'm gonna collect all the image, text and audio files in this folder
output_folder="./data/mixed_data"
output_audio_path ="./data/mixed_data/output_audio.wav"

In [7]:
!mkdir mixed_data //

The syntax of the command is incorrect.


In [ ]:
file_path = output_video_path + "input_video.mp4"
print(file_path)

In [ ]:
def download_video(url, output_path):
    try:
        yt = YouTube(video_url)
        metadata ={
            "title": yt.title,
            "description": yt.description,
            "length": yt.length,
            "views": yt.views,
            "author": yt.author,
            "publish_date": yt.publish_date
        }
        stream = yt.streams.filter(progressive=True, file_extension='mp4').first()
        stream.download(output_path=output_path, filename='input_vid.mp4')
        print(f"Video downloaded successfully to {output_path}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
from moviepy.editor import VideoFileClip
def video_to_images(video_path, output_folder, fps=1):
    try:
        clip = VideoFileClip(video_path)
        clip.write_images_sequence(
            os.path.join(output_folder, "frame%04d.png"), fps=0.2
            )
    except Exception as e:
        print(f"An error occurred while extracting frames: {e}")

In [ ]:
def video_to_audio(video_path, output_audio_path):
    try:
        clip = VideoFileClip(video_path)
        clip.audio.write_audiofile(output_audio_path)
    except Exception as e:
        print(f"An error occurred while extracting audio: {e}")

In [ ]:
def audio_to_text(audio_path):
    try:
        recognizer = sr.Recognizer()
        with sr.AudioFile(audio_path) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_whisper(audio_data)
            return text
    except Exception as e:
        print(f"An error occurred while converting audio to text: {e}")
        return None

In [ ]:
metadata = download_video(video_url, output_video_path)

In [ ]:
video_to_audio(file_path, output_audio_path)

In [ ]:
text_data= audio_to_text(output_audio_path)

In [ ]:
with open (output_folder + "/output_text.txt", "w") as text_file:
    text_file.write(text_data)
print("Text data saved to output_text.txt")
file.close()

In [ ]:
os.remove(output_audio_path)
print("Audio file removed after processing.")

In [ ]:
from llama_index.core.indices import MultiModelVectorStoreIndex
from llama_inex.core import SimpleDirectoryReader, storagecontext
from llama_index.vector_stores.lancedb import LanceDBVectorStore

In [ ]:
text_store= lanceDBVectorStore(Url="lancedb",table_name="tect_collection")
image_store= lanceDBVectorStore(Url="lancedb",table_name="image_collection")

In [ ]:
storage_context = storagecontext.from_defaults(vector_store=text_store, image_store=image_store)

In [ ]:
documents = SimpleDirectoryReader(output_folder).load_data()

In [ ]:
index = MultiModelVectorStoreIndex.from_documents(documents, storage_context=storage_context)

In [ ]:
retriever_engine = index.as_retriever(similarity_top_k=1, image_similarity_top_k=3)

In [ ]:
qa_tmpl_str = (
    "based on the provided information,incuding relevent image and retrieved context from the video"
    "context:{context_str}\n" 
    "metadata:{metadata_str}\n"
    "query: {query_str}"
    "answer:"
    )

SyntaxError: '(' was never closed (1763460511.py, line 1)

In [ ]:
from llama_index.core.response.notebook_utils import display_response
from llama_index.core.Schema import imagenode

In [ ]:
def retrieve(retriever_engine, query_str, metadata_str):
    retriever_results = retriever_engine.retrieve(query_str)

    retrieved_image = []
    retrieved_text = []
    for res_node in retriever_results:
        if isinstance(res_node, imagenode):
            retrieved_image.append(res_node.node.metadata['file_path'])
        else:
            display_response(res_node,source_length=200)
            retrieved_text.append(res_node.text)
    return retrieved_image, retrieved_text


In [ ]:
query = "What is the main topic of the video?"
img,text = retrieve(retriever_engine,query)

In [ ]:
def plot_images(image_path):
    image_shown=0
    plt.figure(figsize=(10, 10))
    for img_path in image_path:
        if os.path.isfile(img_path):
            image = image.open(img_path)
            plt.subplot(1, len(image_path), image_shown + 1)
            plt.imshow(image)
            plt.xticks([])
            plt.yticks([])

            images_shown += 1
            if images_shown >= 5:
                break

In [ ]:
plot_images(img)

In [ ]:
import json
metadata_str = json.dumps(metadata_vid)

In [ ]:
query_str = "What is the main topic of the video?"

In [ ]:
context_str = " ".join(text)

In [ ]:
image_documents = SimpleDirectoryReader(input_files=img).load_data()

In [ ]:
from llamma_index.multi_model_llms.openai import OpenAIMultiModalLLM

In [ ]:
openai_multimodal_llm = OpenAIMultiModalLLM(
    model=" ",
    api_key="api_key",
    max_new_tokens=512)


In [ ]:
result = openai_multimodal_llm.complete(
    prompt=qa_tmpl_str.format(
        query_str=query_str,metadata_str=metadata_str,context_str=context_str
    ),
    image_documents=image_documents,
)

In [ ]:
pprint(result.text)